In [1]:
import pandas as pd
import os

## Get Filtered Counts and Metadata from NextFlow Output
Load in merged gene counts from o2, get DEG analysis.

In [4]:
# Path to your merged RSEM gene counts (edit for each run)
counts_file = "/Users/norawolcott/Documents/Datta Lab/data/bulk RNASeq/260510/rsem.merged.gene_counts.tsv"

# Read the TSV file
counts_df = pd.read_csv(counts_file, sep="\t", index_col=0)

# Show the first 5 rows and first 5 columns
counts_df.iloc[:5, :5]

,transcript_id(s),LIB068971_TRA00321012_n47,LIB068971_TRA00321013_n48,LIB068971_TRA00321014_n49,LIB068971_TRA00321015_n50
gene_id,,,,,
ENSMUSG00000000001,ENSMUST00000000001,2591.0,2545.0,2714.0,3174.0
ENSMUSG00000000003,"ENSMUST00000000003,ENSMUST00000114041",10.0,1.0,4.0,0.0
ENSMUSG00000000028,"ENSMUST00000000028,ENSMUST00000096990,ENSMUST0...",308.0,331.0,325.0,392.0
ENSMUSG00000000031,"ENSMUST00000132294,ENSMUST00000136359,ENSMUST0...",284.0,335.0,324.0,410.0
ENSMUSG00000000037,"ENSMUST00000019101,ENSMUST00000074802,ENSMUST0...",72.0,119.0,98.0,112.0


Load in sample metadata. Note: create and save metadata file as a 2-column tsv file.

In [5]:
# Path to your metadata file (edit for new run)
metadata_file = '/Users/norawolcott/Documents/Datta Lab/data/bulk RNASeq/260510/metadata.tsv'

# Read metadata
metadata_df = pd.read_csv(metadata_file, sep="\t", index_col=0)

# Show first few rows
metadata_df.head()

# Extract sample names from counts columns 
# e.g. "LIB068677_TRA00318992_MOE-c1" > "c1"
counts_df.columns = counts_df.columns.str.extract(r'([a-z0-9]+)$')[0]

Align gene counts with metadata.

In [6]:
# Reorder columns in counts_df to match metadata
counts_df = counts_df[metadata_df.index]

# Quick check: columns now match metadata
print("Counts columns match metadata?", all(counts_df.columns == metadata_df.index))

# Optional: check shape
print("Counts dataframe shape:", counts_df.shape)

Counts columns match metadata? True
Counts dataframe shape: (78334, 19)


Filter low-count genes

In [7]:
# Keep genes with at least 10 counts in at least one sample
filtered_counts = counts_df[counts_df.sum(axis=1) > 10]

# Round all values to nearest integer
filtered_counts_rounded = filtered_counts.round().astype(int)

print("Filtered counts shape:", filtered_counts.shape)

Filtered counts shape: (32861, 19)


Export filtered counts and metadata for DESeq2 in R

In [8]:
# Save filtered counts
filtered_counts_rounded.to_csv("/Users/norawolcott/Documents/Datta Lab/data/bulk RNASeq/260510/filtered_counts.tsv", sep="\t")

# Save metadata
metadata_df.to_csv("/Users/norawolcott/Documents/Datta Lab/data/bulk RNASeq/260510/metadata_for_DE.tsv", sep="\t")

OPTIONAL: concatenate multiple metadata and filtered counts files

In [9]:
# Insert paths (EDIT)
counts_file_1 = '/Users/norawolcott/Documents/Datta Lab/data/bulk RNASeq/260403/filtered_counts.tsv'
counts_file_2 = '/Users/norawolcott/Documents/Datta Lab/data/bulk RNASeq/260510/filtered_counts.tsv'

meta_file_1 = "/Users/norawolcott/Documents/Datta Lab/data/bulk RNASeq/260403/metadata_for_DE.tsv"
meta_file_2 = "/Users/norawolcott/Documents/Datta Lab/data/bulk RNASeq/260510/metadata_for_DE.tsv"

# Load data
counts_1 = pd.read_csv(counts_file_1, sep="\t", index_col=0)
counts_2 = pd.read_csv(counts_file_2, sep="\t", index_col=0)

meta_1 = pd.read_csv(meta_file_1, sep="\t", index_col=0)
meta_2 = pd.read_csv(meta_file_2, sep="\t", index_col=0)

# Concatenate
counts_combined = pd.concat([counts_1, counts_2], axis=1)  # join samples (columns)
counts_combined = counts_combined.fillna(0)  # replace any missing values with 0 (if needed)

meta_combined = pd.concat([meta_1, meta_2], axis=0)        # join samples (rows)

# Save outputs
output_dir = "/Users/norawolcott/Documents/Datta Lab/data/bulk RNASeq/260510/"
os.makedirs(output_dir, exist_ok=True)

counts_combined.to_csv(f"{output_dir}/combined_filtered_counts_0403_0510.tsv", sep="\t")
meta_combined.to_csv(f"{output_dir}/combined_metadata_for_DE_0403_0510.tsv", sep="\t")

print("Done! Files saved to:", output_dir)

Done! Files saved to: /Users/norawolcott/Documents/Datta Lab/data/bulk RNASeq/260510/
